# Balling With AI Group Database

## Congress.Gov

In [ ]:
import requests
import polars as pl
import time

cong_gov_api = "dtsvgW8TZ4HnLUxZaOA5e1PO2RlgZDaCiknU9lMm"

df_base = pl.read_json('bills_111_119.json')
df_base.shape
df_base.columns

['congress',
 'latestAction',
 'number',
 'originChamber',
 'originChamberCode',
 'title',
 'type',
 'updateDate',
 'updateDateIncludingText',
 'url']

Filtering for the subset of features that I need from this prior json file + limiting to 1000 for this restricted dataset due to API limits (API is incredibly slow)

In [19]:
df_filtered = df_base[["type", "number", "congress", "originChamber"]]
df_filtered = df_filtered.tail(1000)
df_filtered.shape
params = {
    "api_key": cong_gov_api,
    "format": "json",
}

Integrating other Congress.Gov Data from this base set

main endpoint

In [ ]:
base_url = "https://api.congress.gov/v3/bill/{congress}/{type}/{number}"
from concurrent.futures import ThreadPoolExecutor


def fetch_bill_data(row):
    c, t, n = row["congress"], row["type"], row["number"]
    url = base_url.format(congress=c, type=t, number=n)
    try:
        resp = requests.get(url, params=params)
        resp.raise_for_status()
        data = resp.json()
        data["congress"] = c
        data["type"] = t
        data["number"] = n
        return data
    except Exception as e:
        print(f"Failed to fetch {c}/{t}/{n}: {e}")
        return None

rows = df_filtered.to_dicts()

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fetch_bill_data, rows))

main_point = [r for r in results if r is not None]
df_results = pl.from_dicts(main_point)

In [27]:
df_results.shape
df_results.write_parquet("main_point.parquet")

actions endpoint

In [29]:
base_url = "https://api.congress.gov/v3/bill/{congress}/{type}/{number}/actions"

def fetch_bill_data(row):
    c, n = row["congress"], row["number"]
    t = row["type"].lower()
    url = base_url.format(congress=c, type=t, number=n)
    try:
        resp = requests.get(url, params=params)
        resp.raise_for_status()
        data = resp.json()
        data["congress"] = c
        data["type"] = t
        data["number"] = n
        return data
    except Exception as e:
        print(f"Failed to fetch {c}/{t}/{n}: {e}")
        return None

rows = df_filtered.to_dicts()

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fetch_bill_data, rows))

actions_point = [r for r in results if r is not None]
df_actions = pl.from_dicts(actions_point)

In [ ]:
df_actions.shape
df_actions.write_parquet("actions_point.parquet")

(1000, 6)

Cosponsors Sub-Endpoint

In [32]:
base_url = "https://api.congress.gov/v3/bill/{congress}/{type}/{number}/cosponsors"

def fetch_bill_data(row):
    c, n = row["congress"], row["number"]
    t = row["type"].lower()
    url = base_url.format(congress=c, type=t, number=n)
    try:
        resp = requests.get(url, params=params)
        resp.raise_for_status()
        data = resp.json()
        data["congress"] = c
        data["type"] = t
        data["number"] = n
        return data
    except Exception as e:
        print(f"Failed to fetch {c}/{t}/{n}: {e}")
        return None

rows = df_filtered.to_dicts()

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fetch_bill_data, rows))

cosponsors_point = [r for r in results if r is not None]
df_cosponsors = pl.from_dicts(cosponsors_point)

In [ ]:
df_cosponsors.shape
df_cosponsors.write_parquet("cosponsors_point.parquet")

cosponsors,pagination,request,congress,type,number
list[struct[11]],struct[3],struct[6],i64,str,str
[],"{0,0,null}","{""7337"",""hr"",""https://api.congress.gov/v3/bill/119/hr/7337?format=json"",""119"",""application/json"",""json""}",119,"""hr""","""7337"""
"[{""F000466"",1,""Brian"",""Rep. Fitzpatrick, Brian K. [R-PA-1]"",true,""Fitzpatrick"",""K."",""R"",""2026-02-03"",""PA"",""https://api.congress.gov/v3/member/F000466?format=json""}, {""M001160"",4,""Gwen"",""Rep. Moore, Gwen [D-WI-4]"",true,""Moore"",null,""D"",""2026-02-03"",""WI"",""https://api.congress.gov/v3/member/M001160?format=json""}, … {""C001080"",28,""Judy"",""Rep. Chu, Judy [D-CA-28]"",false,""Chu"",null,""D"",""2026-02-23"",""CA"",""https://api.congress.gov/v3/member/C001080?format=json""}]","{26,26,""https://api.congress.gov/v3/bill/119/hr/7333/cosponsors?offset=20&limit=20&format=json""}","{""7333"",""hr"",""https://api.congress.gov/v3/bill/119/hr/7333?format=json"",""119"",""application/json"",""json""}",119,"""hr""","""7333"""
[],"{0,0,null}","{""7338"",""hr"",""https://api.congress.gov/v3/bill/119/hr/7338?format=json"",""119"",""application/json"",""json""}",119,"""hr""","""7338"""
"[{""F000466"",1,""Brian"",""Rep. Fitzpatrick, Brian K. [R-PA-1]"",true,""Fitzpatrick"",""K."",""R"",""2026-02-03"",""PA"",""https://api.congress.gov/v3/member/F000466?format=json""}, {""C001121"",6,""Jason"",""Rep. Crow, Jason [D-CO-6]"",true,""Crow"",null,""D"",""2026-02-03"",""CO"",""https://api.congress.gov/v3/member/C001121?format=json""}, … {""M001206"",25,""Joseph"",""Rep. Morelle, Joseph D. [D-NY-25]"",false,""Morelle"",""D."",""D"",""2026-02-20"",""NY"",""https://api.congress.gov/v3/member/M001206?format=json""}]","{5,5,null}","{""7336"",""hr"",""https://api.congress.gov/v3/bill/119/hr/7336?format=json"",""119"",""application/json"",""json""}",119,"""hr""","""7336"""
[],"{0,0,null}","{""7339"",""hr"",""https://api.congress.gov/v3/bill/119/hr/7339?format=json"",""119"",""application/json"",""json""}",119,"""hr""","""7339"""


Committees Sub-Endpoint

In [36]:
base_url = "https://api.congress.gov/v3/bill/{congress}/{type}/{number}/committees"

def fetch_bill_data(row):
    c, n = row["congress"], row["number"]
    t = row["type"].lower()
    url = base_url.format(congress=c, type=t, number=n)
    try:
        resp = requests.get(url, params=params)
        resp.raise_for_status()
        data = resp.json()
        data["congress"] = c
        data["type"] = t
        data["number"] = n
        return data
    except Exception as e:
        print(f"Failed to fetch {c}/{t}/{n}: {e}")
        return None

rows = df_filtered.to_dicts()

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fetch_bill_data, rows))

committees_point = [r for r in results if r is not None]
df_committees = pl.from_dicts(committees_point)

In [37]:
df_committees.shape
df_committees.write_parquet("committees_point.parquet")

Subjects Sub-Endpoint

In [38]:
base_url = "https://api.congress.gov/v3/bill/{congress}/{type}/{number}/subjects"

def fetch_bill_data(row):
    c, n = row["congress"], row["number"]
    t = row["type"].lower()
    url = base_url.format(congress=c, type=t, number=n)
    try:
        resp = requests.get(url, params=params)
        resp.raise_for_status()
        data = resp.json()
        data["congress"] = c
        data["type"] = t
        data["number"] = n
        return data
    except Exception as e:
        print(f"Failed to fetch {c}/{t}/{n}: {e}")
        return None

rows = df_filtered.to_dicts()

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fetch_bill_data, rows))

subjects_point = [r for r in results if r is not None]
df_subjects = pl.from_dicts(subjects_point)

In [39]:
df_subjects.shape
df_subjects.write_parquet("subjects_point.parquet")

Related Bills Sub-Endpoint

In [40]:
base_url = "https://api.congress.gov/v3/bill/{congress}/{type}/{number}/relatedbills"

def fetch_bill_data(row):
    c, n = row["congress"], row["number"]
    t = row["type"].lower()
    url = base_url.format(congress=c, type=t, number=n)
    try:
        resp = requests.get(url, params=params)
        resp.raise_for_status()
        data = resp.json()
        data["congress"] = c
        data["type"] = t
        data["number"] = n
        return data
    except Exception as e:
        print(f"Failed to fetch {c}/{t}/{n}: {e}")
        return None

rows = df_filtered.to_dicts()

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fetch_bill_data, rows))

relatedbills_point = [r for r in results if r is not None]
df_relatedbills = pl.from_dicts(relatedbills_point)

In [41]:
df_relatedbills.shape
df_relatedbills.write_parquet("relatedbills_point.parquet")

MemberEndpoint

In [69]:
base_url= "https://api.congress.gov/v3/member/{bioguideId}"

def fetch_sponsor_details(row):
    bioGuideId= df_results['bill'][0]['sponsors'][0]['bioguideId']
    url = base_url.format(bioguideId=bioGuideId)
    try:
        resp = requests.get(url, params=params)
        resp.raise_for_status()
        data = resp.json()
        data["bioguideId"] = bioGuideId
        return data
    except Exception as e:
        print(f"Failed to fetch sponsor {bioGuideId}: {e}")
        return None

sponsor_rows = df_results.to_dicts()
with ThreadPoolExecutor(max_workers=5) as executor:
    sponsor_details = list(executor.map(fetch_sponsor_details, sponsor_rows))

Member_point = [r for r in sponsor_details if r is not None]
df_member = pl.from_dicts(Member_point)

In [71]:
df_member.shape
df_member.write_parquet("member_point.parquet")

Merge Data Points

In [ ]:
import polars as pl

def prepare_for_join(df, is_base=False):
    """
    Standardizes keys and handles nesting to prevent join failures.
    """
    # 1. Handle Structural Nesting (The DuplicateError fix)
    if "bill" in df.columns:
        # If this isn't our base DF, drop the top-level keys before unnesting
        # to prevent having two 'congress' columns.
        if not is_base:
            cols_to_drop = [c for c in ["congress", "type", "number"] if c in df.columns]
            df = df.drop(cols_to_drop)
        df = df.unnest("bill")

    # 2. Drop common conflicting columns that aren't useful for analysis
    for conflict in ["request", "status_code"]:
        if conflict in df.columns:
            df = df.drop(conflict)

    # 3. Normalize Data Types (The 'everything is null' fix)
    # We cast to String (Utf8) and strip whitespace to ensure '119' == '119'
    return df.with_columns([
        pl.col("congress").cast(pl.Utf8).str.strip_chars(),
        pl.col("type").cast(pl.Utf8).str.strip_chars().str.to_lowercase(),
        pl.col("number").cast(pl.Utf8).str.strip_chars()
    ])

# --- 1. Prepare the DataFrames ---
# Base anchor (Keep these keys!)
df_filtered = prepare_for_join(df_filtered, is_base=True)

# API Results (Unnest and normalize)
df_results      = prepare_for_join(df_results)
df_committees   = prepare_for_join(df_committees)
df_cosponsors   = prepare_for_join(df_cosponsors)
df_relatedbills = prepare_for_join(df_relatedbills)
df_subjects     = prepare_for_join(df_subjects)
df_actions      = prepare_for_join(df_actions)

# --- 2. Execute the Join Chain ---
# We use unique suffixes for each join so you know where the data came from
# (e.g., 'url' becomes 'url_comm' from the committees join)



final_df = (
    df_filtered
    .join(df_results,      on=["congress", "type", "number"], how="left")
    .join(df_committees,   on=["congress", "type", "number"], how="left", suffix="_comm")
    .join(df_cosponsors,   on=["congress", "type", "number"], how="left", suffix="_cospon")
    .join(df_relatedbills, on=["congress", "type", "number"], how="left", suffix="_related")
    .join(df_subjects,     on=["congress", "type", "number"], how="left", suffix="_subj")
    .join(df_actions,      on=["congress", "type", "number"], how="left", suffix="_act")
)

# --- 3. Verify Results ---
print(f"Final Column Count: {len(final_df.columns)}")
print(f"Rows with successful matches: {final_df.filter(pl.col('title').is_not_null()).height}")

Final Column Count: 32
Rows with successful matches: 1000


In [131]:
final_df = final_df.with_columns(
    bioguideId = pl.col("sponsors")
    .list.get(0)               # Grab the first sponsor in the list
    .struct.field("bioguideId") # Reach into that sponsor for the ID
)
final_df.columns

['type',
 'number',
 'congress',
 'originChamber',
 'actions',
 'committees',
 'constitutionalAuthorityStatementText',
 'cosponsors',
 'introducedDate',
 'latestAction',
 'legislationUrl',
 'originChamber_right',
 'originChamberCode',
 'policyArea',
 'relatedBills',
 'sponsors',
 'subjects',
 'textVersions',
 'title',
 'titles',
 'updateDate',
 'updateDateIncludingText',
 'summaries',
 'committeeReports',
 'cboCostEstimates',
 'committees_comm',
 'pagination',
 'cosponsors_cospon',
 'pagination_related',
 'relatedBills_related',
 'subjects_subj',
 'actions_act',
 'bioguideId']

In [145]:
#df_final = final_df.join(df_member, on="bioguideId")
df_final.columns

['type',
 'number',
 'congress',
 'originChamber',
 'actions',
 'committees',
 'constitutionalAuthorityStatementText',
 'cosponsors',
 'introducedDate',
 'latestAction',
 'legislationUrl',
 'originChamber_right',
 'originChamberCode',
 'policyArea',
 'relatedBills',
 'sponsors',
 'subjects',
 'textVersions',
 'title',
 'titles',
 'updateDate',
 'updateDateIncludingText',
 'summaries',
 'committeeReports',
 'cboCostEstimates',
 'committees_comm',
 'pagination',
 'cosponsors_cospon',
 'pagination_related',
 'relatedBills_related',
 'subjects_subj',
 'actions_act',
 'bioguideId',
 'member']

In [108]:
final_df.write_parquet("final_df_conggov.parquet")

# Legislator Info Repo

In [122]:
legislators_url = "https://unitedstates.github.io/congress-legislators/legislators-current.json"
current = requests.get(legislators_url).json()
currentleg_df= pl.from_dicts(current)
currentleg_df = currentleg_df.unnest('id')

In [123]:
currentleg_df.head()

bioguide,thomas,lis,govtrack,opensecrets,votesmart,fec,cspan,wikipedia,house_history,ballotpedia,maplight,icpsr,wikidata,google_entity_id,pictorial,name,bio,terms,leadership_roles,family
str,str,str,i64,str,i64,list[str],i64,str,i64,str,i64,i64,str,str,i64,struct[6],struct[2],list[struct[17]],list[struct[4]],list[struct[2]]
"""C000127""","""00172""","""S275""",300018,"""N00007836""",27122,"[""S8WA00194"", ""H2WA01054""]",26137,"""Maria Cantwell""",10608,"""Maria Cantwell""",544,39310,"""Q22250""","""kg:/m/01x68t""",13398,"{""Maria"",null,""Cantwell"",""Maria Cantwell"",null,null}","{""1958-10-13"",""F""}","[{""rep"",""1993-01-05"",""1995-01-03"",""WA"",""Democrat"",null,null,null,null,null,null,null,null,null,null,1,null}, {""sen"",""2001-01-03"",""2007-01-03"",""WA"",""Democrat"",null,1,""http://cantwell.senate.gov"",null,null,null,null,null,null,null,null,null}, … {""sen"",""2025-01-03"",""2031-01-03"",""WA"",""Democrat"",null,1,""https://www.cantwell.senate.gov"",""511 Hart Senate Office Building Washington DC 20510"",""202-224-3441"",null,""https://www.cantwell.senate.gov/public/index.cfm/email-maria"",""511 Hart Senate Office Building"",""junior"",""http://www.cantwell.senate.gov/public/index.cfm/rss/feed"",null,null}]",null,null
"""K000367""","""01826""","""S311""",412242,"""N00027500""",65092,"[""S6MN00267""]",83701,"""Amy Klobuchar""",16558,"""Amy Klobuchar""",724,40700,"""Q22237""","""kg:/m/05fbpt""",13431,"{""Amy"",""Jean"",""Klobuchar"",""Amy Klobuchar"",null,null}","{""1960-05-25"",""F""}","[{""sen"",""2007-01-04"",""2013-01-03"",""MN"",""Democrat"",null,1,""http://klobuchar.senate.gov/"",""302 HART SENATE OFFICE BUILDING WASHINGTON DC 20510"",""202-224-3244"",""202-228-2186"",""http://www.klobuchar.senate.gov/emailamy.cfm"",""302 Hart Senate Office Building"",null,null,null,null}, {""sen"",""2013-01-03"",""2019-01-03"",""MN"",""Democrat"",null,1,""https://www.klobuchar.senate.gov"",""302 Hart Senate Office Building Washington DC 20510"",""202-224-3244"",""202-228-2186"",""http://www.klobuchar.senate.gov/public/index.cfm/contact"",""302 Hart Senate Office Building"",""senior"",null,null,null}, … {""sen"",""2025-01-03"",""2031-01-03"",""MN"",""Democrat"",null,1,""https://www.klobuchar.senate.gov"",""425 Dirksen Senate Office Building Washington DC 20510"",""202-224-3244"",null,""https://www.klobuchar.senate.gov/public/index.cfm/contact"",""425 Dirksen Senate Office Building"",""senior"",null,null,null}]","[{""Senate Democratic Steering Committee Chair"",""senate"",""2019-01-03"",""2021-01-03""}, {""Senate Democratic Steering Committee Chair"",""senate"",""2021-01-03"",""2023-01-03""}, … {""Senate Democratic Steering Committee Chair"",""senate"",""2025-01-03"",null}]",null
"""S000033""","""01010""","""S313""",400357,"""N00000528""",27110,"[""H8VT01016"", ""S4VT00033""]",994,"""Bernie Sanders""",21173,"""Bernie Sanders""",450,29147,"""Q359442""","""kg:/m/01_gbv""",13382,"{""Bernard"",null,""Sanders"",""Bernard Sanders"",""Bernie"",null}","{""1941-09-08"",""M""}","[{""rep"",""1991-01-03"",""1993-01-03"",""VT"",""Independent"",""Democrat"",null,null,null,null,null,null,null,null,null,0,null}, {""rep"",""1993-01-05"",""1995-01-03"",""VT"",""Independent"",""Democrat"",null,null,null,null,null,null,null,null,null,0,null}, … {""sen"",""2025-01-03"",""2031-01-03"",""VT"",""Independent"",""Democrat"",1,""https://www.sanders.senate.gov"",""332 Dirksen Senate Office Building Washington DC 20510"",""202-224-5141"",null,""https://www.sanders.senate.gov/contact/"",""332 Dirksen Senate Office Building"",""senior"",""http://www.sanders.senate.gov/rss/"",null,null}]","[{""Senate Democratic Outreach Chair"",""senate"",""2023-01-03"",""2025-01-03""}, {""Senate Democratic Outreach Chair"",""senate"",""2025-01-03"",null}]",null
"""W000802""","""01823""","""S316""",412247,"""N00027533""",2572,"[""S6RI00221""]",92235,"""Sheldon Whitehouse""",null,"""Sheldon Whitehouse""",728,40704,"""Q652066""","""kg:/m/07qw94""",13372,"{""Sheldon"",null,"

In [169]:
final_df = final_df.join(currentleg_df, left_on= "bioguideId", right_on="bioguide", how="left", suffix = 'currentleg')

In [149]:
final_df = final_df.drop(['thomas_right',
 'lis_right',
 'govtrack_right',
 'opensecrets_right',
 'votesmart_right',
 'fec_right',
 'cspan_right',
 'wikipedia_right',
 'house_history_right',
 'ballotpedia_right',
 'maplight_right',
 'icpsr_right',
 'wikidata_right',
 'google_entity_id_right',
 'pictorial_right',
 'name_right',
 'bio_right',
 'terms_right',
 'leadership_roles_right',
 'family_right'])

# Voteview Ideology Data

In [152]:
resp = requests.get('https://voteview.com/static/data/out/members/HS119_members.json')
ideology= resp.json()

In [158]:
print(ideology)
ideology_df = pl.from_dicts(ideology, infer_schema_length=1000)
ideology_df.head()

[{'congress': 119, 'chamber': 'House', 'icpsr': 20301, 'state_icpsr': 41, 'district_code': 3, 'state_abbrev': 'AL', 'party_code': 200, 'occupancy': '', 'last_means': '', 'bioname': 'ROGERS, Mike Dennis', 'bioguide_id': 'R000575', 'born': 1958.0, 'died': None, 'nominate_dim1': 0.379, 'nominate_dim2': 0.38, 'nominate_log_likelihood': -5.04124, 'nominate_geo_mean_probability': 0.98524, 'nominate_number_of_votes': '339', 'nominate_number_of_errors': '1', 'conditional': None, 'nokken_poole_dim1': 0.398, 'nokken_poole_dim2': 0.821}, {'congress': 119, 'chamber': 'House', 'icpsr': 21102, 'state_icpsr': 41, 'district_code': 7, 'state_abbrev': 'AL', 'party_code': 100, 'occupancy': '', 'last_means': '', 'bioname': 'SEWELL, Terri', 'bioguide_id': 'S001185', 'born': 1965.0, 'died': None, 'nominate_dim1': -0.402, 'nominate_dim2': 0.394, 'nominate_log_likelihood': -22.41269, 'nominate_geo_mean_probability': 0.93584, 'nominate_number_of_votes': '338', 'nominate_number_of_errors': '11', 'conditional': 

congress,chamber,icpsr,state_icpsr,district_code,state_abbrev,party_code,occupancy,last_means,bioname,bioguide_id,born,died,nominate_dim1,nominate_dim2,nominate_log_likelihood,nominate_geo_mean_probability,nominate_number_of_votes,nominate_number_of_errors,conditional,nokken_poole_dim1,nokken_poole_dim2
i64,str,i64,i64,i64,str,i64,str,str,str,str,f64,f64,f64,f64,f64,f64,str,str,null,f64,f64
119,"""House""",20301,41,3,"""AL""",200,"""""","""""","""ROGERS, Mike Dennis""","""R000575""",1958.0,null,0.379,0.38,-5.04124,0.98524,"""339""","""1""",null,0.398,0.821
119,"""House""",21102,41,7,"""AL""",100,"""""","""""","""SEWELL, Terri""","""S001185""",1965.0,null,-0.402,0.394,-22.41269,0.93584,"""338""","""11""",null,-0.379,0.331
119,"""House""",21500,41,6,"""AL""",200,"""""","""""","""PALMER, Gary James""","""P000609""",1954.0,null,0.674,0.116,-22.05007,0.93774,"""343""","""13""",null,0.539,-0.032
119,"""House""",22140,41,1,"""AL""",200,"""""","""""","""MOORE, Barry""","""M001212""",1966.0,null,0.643,-0.247,-35.09857,0.90137,"""338""","""16""",null,0.663,-0.257
119,"""House""",22366,41,5,"""AL""",200,"""""","""""","""STRONG, Dale""","""S001220""",1970.0,null,0.602,0.368,-14.53147,0.9584,"""342""","""10""",null,0.427,0.756


In [159]:
final_df = final_df.join(ideology_df, left_on= "bioguideId", right_on="bioguide_id")

In [166]:
final_df.head()

type,number,congress,originChamber,actions,committees,constitutionalAuthorityStatementText,cosponsors,introducedDate,latestAction,legislationUrl,originChamber_right,originChamberCode,policyArea,relatedBills,sponsors,subjects,textVersions,title,titles,updateDate,updateDateIncludingText,summaries,committeeReports,cboCostEstimates,committees_comm,pagination,cosponsors_cospon,pagination_related,relatedBills_related,subjects_subj,actions_act,bioguideId,thomas,lis,govtrack,opensecrets,votesmart,fec,cspan,wikipedia,house_history,ballotpedia,maplight,icpsr,wikidata,google_entity_id,pictorial,name,bio,terms,leadership_roles,family,congress_right,chamber,icpsr_right,state_icpsr,district_code,state_abbrev,party_code,occupancy,last_means,bioname,born,died,nominate_dim1,nominate_dim2,nominate_log_likelihood,nominate_geo_mean_probability,nominate_number_of_votes,nominate_number_of_errors,conditional,nokken_poole_dim1,nokken_poole_dim2
str,str,str,str,struct[2],struct[2],str,struct[3],str,struct[3],str,str,str,struct[1],struct[2],list[struct[10]],struct[2],struct[2],str,struct[2],str,str,struct[2],list[struct[2]],list[struct[4]],list[struct[7]],struct[1],list[struct[11]],struct[1],list[struct[7]],struct[2],list[struct[9]],str,str,str,i64,str,i64,list[str],i64,str,i64,str,i64,i64,str,str,i64,struct[6],struct[2],list[struct[17]],list[struct[4]],list[struct[2]],i64,str,i64,i64,i64,str,i64,str,str,str,f64,f64,f64,f64,f64,f64,str,str,null,f64,f64
"""hr""","""7337""","""119""","""House""","{3,""https://api.congress.gov/v3/bill/119/hr/7337/actions?format=json""}","{1,""https://api.congress.gov/v3/bill/119/hr/7337/committees?format=json""}","""<pre> [Congressional Record Vo…",null,"""2026-02-03""","{""2026-02-03"",null,""Referred to the House Committee on House Administration.""}","""https://www.congress.gov/bill/…","""House""","""H""","{""Congress""}","{1,""https://api.congress.gov/v3/bill/119/hr/7337/relatedbills?format=json""}","[{""S001215"",11,""Haley"",""Rep. Stevens, Haley M. [D-MI-11]"",""N"",""Stevens"",""M."",""D"",""MI"",""https://api.congress.gov/v3/member/S001215?format=json""}]","{1,""https://api.congress.gov/v3/bill/119/hr/7337/subjects?format=json""}","{1,""https://api.congress.gov/v3/bill/119/hr/7337/text?format=json""}","""Make Congress Drive Union-Made…","{3,""https://api.congress.gov/v3/bill/119/hr/7337/titles?format=json""}","""2026-02-25T17:19:29Z""","""2026-02-25T17:19:29Z""",null,null,null,"[{[{""2026-02-03T15:00:45Z"",""Referred To""}],""House"",""Committee on House Administration"",null,""hsha00"",""Standing"",""https://api.congress.gov/v3/committee/house/hsha00?format=json""}]",{1},[],{1},"[{119,{""2026-02-03"",null,""Read twice and referred to the Committee on Homeland Security and Governmental Affairs.""},3766,[{""CRS"",""Related bill""}],""Make Congress Drive Union Made Act"",""S"",""https://api.congress.gov/v3/bill/119/s/3766?format=json""}]","{[],{""Congress"",""2026-02-13T16:46:41Z""}}","[{""H11100"",""2026-02-03"",null,null,{2,""House floor actions""},""Referred to the House Committee on House Administration."",""IntroReferral"",null,[{""Committee on House Administration"",""hsha00"",""https://api.congress.gov/v3/committee/house/hsha00?format=json""}]}, {""Intro-H"",""2026-02-03"",null,null,{9,""Library of Congress""},""Introduced in House"",""IntroReferral"",null,null}, {""1000"",""2026-02-03"",null,null,{9,""Library of Congress""},""Introduced in House"",""IntroReferral"",null,null}]","""S001215""",null,null,412786,"""N00040915""",181092,"[""H8MI11254""]",null,"""Haley Stevens""",null,"""Haley Stevens""",null,21972,"""Q58322563""","""kg:/g/11fjw095g8""",13104,"{""Haley"",""M."",""Stevens"",""Haley M. Stevens"",null,null}","{""1983-06-24"",""F""}","[{""rep"",""2019-01-03"",""2021-01-03"",""MI"",""Democrat"",null,null,""https://stevens.house.gov"",""227 Cannon House Office Building Washington DC 20515-2211"",""202-225-8171"",null,null,""227 Cannon House Office Building"",null,null,11,null}, {""rep"",""2021-01-03"","